# Fig. 1d — Feature plots of VHC type and striolar/extrastriolar marker genes

**Input**
- `adata_all_final.h5ad` (merged VGN + VHC AnnData with `batch`, `all_cluster_gene_name`, `location` in `.obs`)

**Output**
- `figures/Fig1d_VHC_marker_<gene>.pdf` — one feature plot per marker (greyscale)
- `figures/Fig1d_VHC_location.pdf` — UMAP coloured by anatomical `location` (utricle / saccule / crista)

**Notes**
- VHC cells are obtained by excluding `batch == 'neuron'` from the merged adata.
- Marker-specific `vmin/vmax` percentiles are tuned per gene to emphasise the strongest-expressing cluster while keeping background low; values reproduce those used in the published figure.
- Greyscale colormap is `Greys` truncated to the white-to-dark-grey portion to avoid pure black saturation.

In [ ]:
from pathlib import Path
import numpy as np
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(5, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

## 1. Load adata and subset to VHCs

In [ ]:
adata_all = sc.read_h5ad(DATA_DIR / 'adata_all_final.h5ad')
adata_HC = adata_all[~adata_all.obs['batch'].isin(['neuron'])].copy()
print(adata_HC)

## 2. Greyscale colormap helper
Truncated `Greys` (0 → 82% of dark grey) avoids saturating high-expressing cells to pure black.

In [ ]:
def truncated_greys(top=0.82, n=256):
    return mcolors.LinearSegmentedColormap.from_list(
        'white_darkgrey', cm.Greys(np.linspace(0.0, top, n))
    )

## 3. Feature plots — VHC type + striolar/extrastriolar markers
Per-gene contrast tuning (`vmin / vmax` percentiles, colormap truncation) follows the published figure.

In [ ]:
# (gene, vmin_pct, vmax_pct, cmap_top): per-marker contrast tuning
MARKER_PARAMS = [
    ('Ocm',    'p60', 'p95', 0.89),  # striolar Type I
    ('Bmp2',   'p40', 'p75', 0.79),  # striolar
    ('Spp1',   'p30', 'p55', 0.79),  # extrastriolar
    ('Sox2',   'p20', 'p61', 0.79),  # Type II
    ('Kcnh6',  'p30', 'p75', 0.79),  # Type I
    ('Adam11', 'p30', 'p55', 0.79),
]

def plot_marker(adata, gene, vmin, vmax, cmap_top, save_path):
    cmap = truncated_greys(top=cmap_top)
    plt.figure(figsize=(20, 10))
    sc.pl.umap(
        adata,
        color=[gene],
        add_outline=True,
        legend_fontsize=8,
        legend_fontoutline=2,
        frameon=False,
        cmap=cmap,
        size=60,
        legend_loc='right margin',
        legend_fontweight='semibold',
        show=False,
        layer='log_norm',
        vmax=vmax,
        vmin=vmin,
    )
    ax = plt.gca()
    for coll in ax.collections:
        if hasattr(coll, 'set_alpha'):
            coll.set_alpha(1)
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

for gene, vmin, vmax, cmap_top in MARKER_PARAMS:
    plot_marker(
        adata_HC, gene, vmin, vmax, cmap_top,
        FIG_DIR / f'Fig1d_VHC_marker_{gene}.pdf',
    )

## 4. Anatomical location panel (utricle / saccule / crista)
Categorical UMAP for the same VHC subset. Palette is hand-picked to match the published figure.

In [ ]:
# Palette order must align with adata_HC.obs['location'].cat.categories.
location_colors = ['#DEAC80', '#FBFBFA', '#914F1E']
adata_HC.uns['location_colors'] = location_colors

sc.pl.umap(
    adata_HC,
    color=['location'],
    add_outline=True,
    legend_fontsize=8,
    legend_fontoutline=2,
    frameon=False,
    size=80,
    legend_loc='right margin',
    legend_fontweight='semibold',
    show=False,
)
ax = plt.gca()
for coll in ax.collections:
    if hasattr(coll, 'set_alpha'):
        coll.set_alpha(1)
plt.savefig(FIG_DIR / 'Fig1d_VHC_location.pdf', bbox_inches='tight', dpi=300)
plt.show()